In [11]:
import torch
import requests
from PIL import Image
from torchvision import transforms
from transformers import XLMRobertaTokenizer
import torch.nn.functional as F
import pandas as pd
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "4"  # GPU 사용 설정 (필요시 수정)

# BEiT-3 저장소의 필요한 클래스와 설정 함수를 직접 가져옵니다.
from modeling_finetune import BEiT3_Binary_Classification
from modeling_utils import _get_large_config

# --- 1. 설정 및 모델, 토크나이저 로드 ---

# 사용자 환경에 맞게 경로를 수정해주세요.
BASE_DIR = "/data/2_data_server/cv-07/dice/2025_samsung_challenge"
MODEL_DIR = os.path.join(BASE_DIR, "finetuning/unilm/beit3/")
DATA_DIR = os.path.join(BASE_DIR, "data/data_2/")
# /data/2_data_server/cv-07/dice/2025_samsung_challenge/finetuning/unilm/beit3/output_binary_classification/checkpoint-best.pth

TOKENIZER_PATH = os.path.join(BASE_DIR, "unilm/beit3/beit3.spm")
MODEL_WEIGHTS_PATH = os.path.join(MODEL_DIR, "output_binary_classification","checkpoint-best.pth")
TEST_CSV_PATH = os.path.join(DATA_DIR, "test.csv")
IMAGE_DIR = os.path.join(DATA_DIR, "test_input_images")
OUTPUT_CSV_PATH = os.path.join(DATA_DIR, "zeroshot_binary_statement .csv")


# 디바이스 설정 (GPU 우선 사용)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 토크나이저 로드
tokenizer = XLMRobertaTokenizer(TOKENIZER_PATH)
print("✅ Tokenizer 로드 완료")

# 필요한 토큰 ID 가져오기
true_token_id = tokenizer.convert_tokens_to_ids("true")
false_token_id = tokenizer.convert_tokens_to_ids("false")
mask_token_id = tokenizer.mask_token_id
print(f"ID: true={true_token_id}, false={false_token_id}, [MASK]={mask_token_id}")

# --- 모델 로드 ---

args = _get_large_config(img_size=224)
args.vocab_size = 64010
# args.num_classes = 2  # BEiT-3는 다중 클래스 분류를 지원하므로, 이진 분류를 위해 num_classes를 2로 설정합니다.
print(f"✅ 모델 설정(args) 생성 완료 (img_size=224, vocab_size=64010)")

model = BEiT3_Binary_Classification(args=args, num_classes=2)
# model = BEiT3_Binary_Classification()
print("✅ 모델 구조 생성 완료")

# 1. 'large' 모델 가중치를 로드할 때, weights_only=False 인자를 추가합니다.
state_dict = torch.load(MODEL_WEIGHTS_PATH, map_location='cpu', weights_only=False)["model"]
# ▲▲▲ 수정 완료 ▲▲▲

# 2. 불필요한 'mim_head' 가중치를 state_dict에서 제거합니다.
keys_to_delete = [key for key in state_dict if key.startswith('mim_head')]
for key in keys_to_delete:
    del state_dict[key]
print(f"✅ 불필요한 가중치 키 제거 완료: {keys_to_delete}")


model.load_state_dict(state_dict, strict=False)
model.to(device)
model.eval()
print("✅ 모델 가중치 로드 및 초기화 완료")


# --- 2. 이미지 전처리 ---
transform = transforms.Compose([
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
])

# --- 3. 제로샷 예측 함수 ---
def zero_shot_true_false_prediction(image_path, question, choice, model, tokenizer, transform, device):
    try:
        image = Image.open(image_path).convert("RGB")
        image_tensor = transform(image).unsqueeze(0).to(device)
    except Exception as e:
        print(f"이미지를 로드할 수 없습니다: {image_path} - {e}")
        return None

    text_prompt = f"{question} {choice} this statement is {tokenizer.mask_token}"
    encoding = tokenizer(text_prompt, return_tensors="pt")
    input_ids = encoding.input_ids.to(device)
    attention_mask = encoding.attention_mask.to(device)

    mask_token_indices = torch.where(input_ids == mask_token_id)[1]
    if len(mask_token_indices) == 0:
        print(f"프롬프트에서 [MASK] 토큰을 찾을 수 없습니다: {text_prompt}")
        return None

    with torch.no_grad():
        outputs = model.beit3(
            textual_tokens=input_ids,
            visual_tokens=image_tensor,
            text_padding_position=attention_mask.bool().logical_not()
        )
        encoder_out = outputs["encoder_out"]
        batch_indices = torch.arange(encoder_out.size(0))
        masked_token_feats = encoder_out[batch_indices, mask_token_indices]
        mlm_logits = model.mlm_head(masked_token_feats)
        true_false_logits = mlm_logits[:, [true_token_id, false_token_id]]
        probabilities = F.softmax(true_false_logits, dim=-1).squeeze()

        if probabilities.numel() == 2:
            return probabilities[0].item() # 'true'일 확률만 반환
        else:
            return None

✅ Tokenizer 로드 완료
ID: true=24216, false=27882, [MASK]=64001
✅ 모델 설정(args) 생성 완료 (img_size=224, vocab_size=64010)
✅ 모델 구조 생성 완료
✅ 불필요한 가중치 키 제거 완료: []


RuntimeError: Error(s) in loading state_dict for BEiT3_Binary_Classification:
	size mismatch for beit3.text_embed.weight: copying a param with shape torch.Size([64010, 768]) from checkpoint, the shape in current model is torch.Size([64010, 1024]).
	size mismatch for beit3.vision_embed.mask_token: copying a param with shape torch.Size([1, 1, 768]) from checkpoint, the shape in current model is torch.Size([1, 1, 1024]).
	size mismatch for beit3.vision_embed.cls_token: copying a param with shape torch.Size([1, 1, 768]) from checkpoint, the shape in current model is torch.Size([1, 1, 1024]).
	size mismatch for beit3.vision_embed.proj.weight: copying a param with shape torch.Size([768, 3, 16, 16]) from checkpoint, the shape in current model is torch.Size([1024, 3, 16, 16]).
	size mismatch for beit3.vision_embed.proj.bias: copying a param with shape torch.Size([768]) from checkpoint, the shape in current model is torch.Size([1024]).
	size mismatch for beit3.encoder.embed_positions.A.weight: 

In [ ]:


# --- 4. 메인 실행 로직: CSV 읽고 정답 예측 및 저장 ---

try:
    df = pd.read_csv(TEST_CSV_PATH)
except FileNotFoundError:
    print(f"오류: 테스트 파일을 찾을 수 없습니다. 경로를 확인하세요: {TEST_CSV_PATH}")
    exit()

results = []
options = ['A', 'B', 'C', 'D']

for index, row in df.iterrows():
    test_id = row['ID']
    question = row['Question']
    image_filename = os.path.basename(row['img_path'])
    image_path = os.path.join(IMAGE_DIR, image_filename)

    best_option = None
    max_true_prob = -1.0

    print(f"\n--- ID: {test_id} 처리 중 ---")
    print(f"이미지: {image_path}")

    for option_letter in options:
        choice_text = row[option_letter]
        true_prob = zero_shot_true_false_prediction(image_path, question, choice_text, model, tokenizer, transform, device)

        if true_prob is not None:
            print(f"  보기 {option_letter}: '{choice_text[:30]}...' -> True 확률: {true_prob:.4f}")
            if true_prob > max_true_prob:
                max_true_prob = true_prob
                best_option = option_letter

    print(f"-> 최종 선택: {best_option} (확률: {max_true_prob:.4f})")
    results.append({'ID': test_id, 'answer': best_option})

# --- 요청하신 형식으로 CSV 파일 저장 ---
submission_df = pd.DataFrame(results)
submission_df.to_csv(OUTPUT_CSV_PATH, index=False)

print(f"\n✅ 모든 작업 완료! 예측 결과가 '{OUTPUT_CSV_PATH}' 파일에 저장되었습니다.")

✅ Tokenizer 로드 완료
ID: true=24216, false=27882, [MASK]=64001
✅ 모델 설정(args) 생성 완료 (img_size=224, vocab_size=64010)
✅ 모델 구조 생성 완료
✅ 불필요한 가중치 키 제거 완료: ['mim_head.weight', 'mim_head.bias']
✅ 모델 가중치 로드 및 초기화 완료

--- ID: TEST_000 처리 중 ---
이미지: /data/2_data_server/cv-07/dice/2025_samsung_challenge/data/data_2/test_input_images/TEST_000.jpg
  보기 A: 'Bananas and grapes placed in b...' -> True 확률: 0.7266
  보기 B: 'Apples and oranges displayed o...' -> True 확률: 0.8234
  보기 C: 'Peaches and plums in a wooden ...' -> True 확률: 0.5897
  보기 D: 'Pears and lemons arranged neat...' -> True 확률: 0.6391
-> 최종 선택: B (확률: 0.8234)

--- ID: TEST_001 처리 중 ---
이미지: /data/2_data_server/cv-07/dice/2025_samsung_challenge/data/data_2/test_input_images/TEST_001.jpg
  보기 A: 'Chinese, famous for dumplings ...' -> True 확률: 0.2925
  보기 B: 'Mexican, featuring tacos and e...' -> True 확률: 0.6755
  보기 C: 'Italian, known for pasta dishe...' -> True 확률: 0.5736
  보기 D: 'Indian, rich in spices and cur...' -> True 확률: 0.5372
-> 최종 선택

: 